# Load data

In [5]:
import pandas as pd

In [6]:
df = pd.read_pickle(r"../../../data/preprocessed/stemm_lemm_stop_words_stop_fb_pages_2016.pkl")


In [7]:
df.shape

(62844, 17)

In [8]:
df.head()

,Facebook Id,Post Created,Total Interactions,Likes,Comments,Shares,Love,Wow,Haha,Sad,Angry,Care,Message,Message_stpWrd,Message_clean_stpWrd,Message_clean_stemm_stpWrd,Message_clean_lemm_stpWrd
0,129348553750248,2016-12-31 23:56:34 CST,17,13,0,4,0,0,0,0,0,0,"FELIZ AÑO NUEVO! Belice, Costa Rica, El Salvad...","FELIZ AÑO NUEVO! Belice, Costa Rica, Salvador,...",feliz ano belice costa rica salvador guatemala...,feliz ano nuev belic cost ric salvador guatema...,feliz ano belice costa rico salvador guatemala...
1,129348553750248,2016-12-31 23:50:53 CST,2,2,0,0,0,0,0,0,0,0,MENSAJE DE PRESERVE PLANET PARA EL NUEVO AÑO. ...,"MENSAJE PRESERVE PLANET AÑO. 📃 Iniciamos 2017,...",mensaje preserve planet ano iniciamos 2017 med...,mensaj preserv planet par nuev ano inici 2017 ...,mensaje preserve planet ano iniciar 2017 medio...
2,100067845290450,2016-12-31 23:13:19 CST,19,15,0,3,1,0,0,0,0,0,Un #FelizAño les desea el Instituto Distrital ...,#FelizAño desea Instituto Distrital Gestión Ri...,felizano desea instituto distrital gestion rie...,felizan dese institut distrital gestion riesg ...,felizano desear instituto distrital gestion ri...
3,100070452191560,2016-12-31 22:40:00 CST,5,3,0,2,0,0,0,0,0,0,"Francia busca siempre ""con el diálogo solucion...","Francia busca ""con diálogo soluciones"", Oriente",francia busca dialogo soluciones tambien oriente,franci busc siempr dialog solucion tambi orient,francia buscar dialogo solución tambien oriente
4,129348553750248,2016-12-31 22:31:54 CST,6,5,0,0,1,0,0,0,0,0,FELIZ AÑO NUEVO! ⤵️ MENSAJE DE PRESERVE PLANET...,FELIZ AÑO NUEVO! ⤵️ MENSAJE PRESERVE PLANET 20...,feliz ano mensaje preserve planet 2017 iniciam...,feliz ano nuev mensaj preserv planet par 2017 ...,feliz ano mensaje preserve planet 2017 iniciar...


In [9]:
df.columns

Index(['Facebook Id', 'Post Created', 'Total Interactions', 'Likes',
       'Comments', 'Shares', 'Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care',
       'Message', 'Message_stpWrd', 'Message_clean_stpWrd',
       'Message_clean_stemm_stpWrd', 'Message_clean_lemm_stpWrd'],
      dtype='object')

In [10]:
selected_data = df[['Message' ,'Message_stpWrd', 'Message_clean_lemm_stpWrd']]

In [11]:
selected_data.head()

,Message,Message_stpWrd,Message_clean_lemm_stpWrd
0,"FELIZ AÑO NUEVO! Belice, Costa Rica, El Salvad...","FELIZ AÑO NUEVO! Belice, Costa Rica, Salvador,...",feliz ano belice costa rico salvador guatemala...
1,MENSAJE DE PRESERVE PLANET PARA EL NUEVO AÑO. ...,"MENSAJE PRESERVE PLANET AÑO. 📃 Iniciamos 2017,...",mensaje preserve planet ano iniciar 2017 medio...
2,Un #FelizAño les desea el Instituto Distrital ...,#FelizAño desea Instituto Distrital Gestión Ri...,felizano desear instituto distrital gestion ri...
3,"Francia busca siempre ""con el diálogo solucion...","Francia busca ""con diálogo soluciones"", Oriente",francia buscar dialogo solución tambien oriente
4,FELIZ AÑO NUEVO! ⤵️ MENSAJE DE PRESERVE PLANET...,FELIZ AÑO NUEVO! ⤵️ MENSAJE PRESERVE PLANET 20...,feliz ano mensaje preserve planet 2017 iniciar...


# spaCy

We are going to use `spaCy`'s pretrained models to extract all organizations mentioned on the posts

In [12]:
import spacy
from spacy import displacy
from collections import Counter

In [13]:
spacy.__version__

'3.7.4'

In [14]:
nlp = spacy.load('es_core_news_lg')

We will use `spaCy` models to identify and extract organizations from posts: 

In [15]:
doc = nlp(selected_data.iloc[4183]['Message'])

In [16]:
displacy.serve(doc, style="ent", auto_select_port=True)

/opt/anaconda3/envs/nanook_env/lib/python3.9/site-packages/spacy/util.py:1835: UserWarning: [W124] 0.0.0.0:5000 is already in use, using the nearest available port 5001 as an alternative.
  warnings.warn(Warnings.W124.format(host=host, port=start, serve_port=port))
/opt/anaconda3/envs/nanook_env/lib/python3.9/site-packages/spacy/displacy/__init__.py:106: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'ent' visualizer
Serving on http://0.0.0.0:5001 ...

Shutting down server on port 5001.


In [ ]:
doc.ents

Generar función para extraer todas las organizaciones mencionadas en cada un post de Facebook.

In [15]:
def get_orgs(text):
    doc = nlp(text)
    org_list = []
    for entity in doc.ents:
        if entity.label_ == 'ORG':
            org_list.append(entity.text)
    org_list = list(set(org_list))
    return org_list

The next cell takes around $4$ hours to extract all organizations...

In [ ]:
if 1 == 0:
    selected_data['organizations'] = selected_data['Message'].apply(get_orgs)
    selected_data.to_pickle(r"../../../data/preprocessed/full_data_organizations_2016.pkl")
else:
    selected_data = pd.read_pickle(r"../../../data/preprocessed/full_data_organizations_2016.pkl")
selected_data.head()

Let's extract all organizations:

In [19]:
orgs = selected_data['organizations'].to_list()
orgs = [org for sublist in orgs for org in sublist]

In [ ]:
org_freq = Counter(orgs)
org_freq.most_common(20)

What about the following post?

In [ ]:
print(selected_data.iloc[0]['Message'])

In [ ]:
selected_data.iloc[0]['organizations']

# References
- [Intro to `spaCy`](https://spacy.io/usage/spacy-101/)
- [spaCy NER](https://spacy.io/usage/linguistic-features#named-entities)
- [spaCy visualizaition](https://spacy.io/usage/visualizers)

# FuzzyWuzzy


In [19]:
selected_data.shape

(62844, 3)

In [17]:
sin_duplicados = selected_data.drop_duplicates()

In [18]:
sin_duplicados.shape

(58492, 3)

In [ ]:
selected_data.columns

In [ ]:
selected_data.shape

In [31]:
from deduplication import remove_similar_messages_parallel, analyze_duplicates

In [ ]:
test_subset = selected_data.head(1000)
test_dedup = remove_similar_messages_parallel(test_subset)
analyze_duplicates(test_subset, test_dedup)

In [ ]:
deduplicated_df = remove_similar_messages_parallel(selected_data)

# Analyze the results
analyze_duplicates(selected_data, deduplicated_df)

# save the deduplicated DataFrame
deduplicated_df.to_csv('deduplicated_2016.csv', index=False)

In [ ]:
print(selected_data["Message"].head(20))
print(deduplicated_df["Message"].head(20))



In [ ]:
# PRUEBA
test_data = pd.DataFrame({
    'Message': [
        "This is a test message",
        "This is a test message!", 
        "This is a completely different message",
        "This is a tst message",   
        "Hello world"
    ],
    'Message_stpWrd': ['dummy1', 'dummy2', 'dummy3', 'dummy4', 'dummy5'],
    'Message_clean_lemm_stpWrd': ['dummy1', 'dummy2', 'dummy3', 'dummy4', 'dummy5'],
    'organizations': ['org1', 'org2', 'org3', 'org4', 'org5']
})


deduplicated_df = remove_similar_messages_parallel(test_data)


print("\nOriginal DataFrame:")
print(test_data['Message'])
print("\nDeduplicated DataFrame:")
print(deduplicated_df['Message'])